# 02 · Memory — 02 long-term memory (a decision that survives the process)

**Long-term memory is a decision written down somewhere a different process can read it.** That is the entire mechanism. Everything else — stores, embeddings, retention policies — is detail on top of "write it to a file, read it back later."

This notebook ports TerrierTA's deduction memory (`src/terrier_ta/services/deduction_memory.py`, plus the consistency check from `services/memory_hooks.py`). It exists to solve a specific problem in automated grading: **the same mistake should cost the same marks every time it appears**, whether that is the second submission in a batch or the first one next semester. An LLM grader asked cold will happily deduct 2 marks for a missing unit on Monday and 5 on Tuesday, and be confident both times. Writing the first decision down and reading it back is what makes the second one consistent.

Everything here writes to a **local file** under this repo's `runs/` directory. No S3, no database, no account anywhere.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `store_path` | Resolves the local JSON file for a course, creating the directory. | `store_path("PHYS101")` → `runs/memory/PHYS101/deductions.json` |
| `save_deduction` | Appends a graded error, or merges it into an existing entry of the same type and bumps `count`. | `save_deduction("PHYS101", error_type="missing units", points_deducted=2.0, ...)` |
| `find_similar_deductions` | Substring-matches a query against stored `error_type` + `reasoning`. | `find_similar_deductions("PHYS101", "units")` → `[{...}]` |
| `apply_deduction` | The consistency hook: warns when a new decision disagrees with the remembered one by more than a mark, then saves. | `apply_deduction("PHYS101", "missing units", 5.0, ...)` |
| `grade_with_memory` | A deterministic stand-in grader that consults memory before deciding a penalty. | `grade_with_memory("PHYS101", "missing units")` → `2.0` |

## Step 1 — bootstrap the repo path and confirm the environment

The same walk-up every notebook in this repo opens with. No key is needed anywhere in this notebook — everything below is stdlib `json` and the filesystem.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

# bootstrap() above already put the repo root on sys.path and loaded .env;
# `_root` is that same path, reused here rather than resolved a second time.
repo_root = _root
env = nbio.show_environment()

## Step 2 — where the memory lives: a local JSON file

`store_path` is TerrierTA's `_store_path` with the tenant lookup replaced by a plain `course_id` and the settings object replaced by `nbio.runs_dir()`. `runs/` is gitignored, so nothing written here can be committed by accident.

In [ ]:
import json
from pathlib import Path


def store_path(course_id: str) -> Path:
    """The local JSON file holding one course's remembered deductions."""
    base = nbio.runs_dir() / "memory" / course_id
    base.mkdir(parents=True, exist_ok=True)
    return base / "deductions.json"


COURSE = "DEMO-PHYS101"

path = store_path(COURSE)
# Start from a clean slate so this notebook's assertions hold on a re-run.
if path.is_file():
    path.unlink()

print(f"store file : {path.relative_to(repo_root)}")
print(f"exists yet : {path.is_file()}")

## Step 3 — reading memory that does not exist yet

The read path has to survive a missing file and a corrupt one. Both return an empty list rather than raising: a grader with no memory should grade, not crash. `MAX_ITEMS` is the port's cap, `max(50, configured)`.

In [ ]:
MAX_ITEMS = max(50, 500)  # the port's floor-and-default cap


def find_similar_deductions(course_id: str, query: str, *, max_results: int = 10) -> list[dict]:
    """Substring search over remembered deductions for one course.

    Matches `query` against `error_type` + `reasoning`, lowercased. An empty
    query matches everything.
    """
    p = store_path(course_id)
    if not p.is_file():
        return []
    try:
        data = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        return []
    items = data if isinstance(data, list) else data.get("deductions", [])
    q = (query or "").lower()
    out = []
    for item in items:
        if not isinstance(item, dict):
            continue
        hay = f"{item.get('error_type', '')} {item.get('reasoning', '')}".lower()
        if not q or q in hay:
            out.append(item)
        if len(out) >= max_results:
            break
    return out


empty = find_similar_deductions(COURSE, "units")
print(f"find_similar_deductions on a missing file -> {empty!r}")
assert empty == [], "a missing store must read as empty, not raise"

## Step 4 — the write path

`save_deduction` ported as written. Two behaviours worth naming now, because both come back later in this notebook:

- **merge on duplicate `error_type`** — a second decision about the same error type does not append a second entry; it *overwrites* `points_deducted` and `reasoning` on the existing one and increments `count`;
- **the cap is applied as `items[-cap:]`** — oldest-inserted entries go first.

In [ ]:
def save_deduction(
    course_id: str,
    *,
    error_type: str,
    points_deducted: float,
    reasoning: str,
    instructor_id: str = "",
) -> None:
    """Record one graded error, merging into an existing entry of the same type."""
    p = store_path(course_id)
    items: list[dict] = []
    if p.is_file():
        try:
            raw = json.loads(p.read_text(encoding="utf-8"))
            items = raw if isinstance(raw, list) else list(raw.get("deductions") or [])
        except Exception:
            items = []

    # Same error type seen before: update the entry in place and count it again.
    for item in items:
        if str(item.get("error_type") or "").lower() == error_type.lower():
            item["points_deducted"] = points_deducted
            item["reasoning"] = reasoning
            item["count"] = int(item.get("count") or 1) + 1
            break
    else:
        items.append(
            {
                "error_type": error_type,
                "points_deducted": points_deducted,
                "reasoning": reasoning,
                "instructor_id": instructor_id,
                "count": 1,
            }
        )

    if len(items) > MAX_ITEMS:
        items = items[-MAX_ITEMS:]

    p.write_text(json.dumps({"deductions": items}, ensure_ascii=False, indent=2), encoding="utf-8")

## Step 5 — session one: grade three submissions and write the decisions down

Synthetic submissions, invented for this notebook. No student, no course, no rubric from anywhere real.

The penalties here are the *first* judgment on each error type — whoever or whatever produced them, this is the moment the number gets fixed.

In [ ]:
session_one = [
    ("missing units", 2.0, "Final answer given as 9.8 with no unit; physical quantity must carry one."),
    ("sign error in integration", 3.0, "Sign flipped at the bounds substitution, propagating to the final value."),
    ("unlabelled free-body diagram", 1.0, "Diagram drawn correctly but forces not labelled."),
]

for error_type, pts, reason in session_one:
    save_deduction(COURSE, error_type=error_type, points_deducted=pts, reasoning=reason, instructor_id="demo-instructor")

stored = json.loads(path.read_text())
nbio.show_json(stored)
print()
print(f"file size on disk: {path.stat().st_size} bytes")
assert len(stored["deductions"]) == 3

## Step 6 — session one ends

Everything the kernel held about those decisions is deleted. What remains is a file.

In [ ]:
del session_one, stored
print("in-kernel state for session one: deleted")
print(f"on disk: {path.is_file()}  ({path.stat().st_size} bytes)")

## Step 7 — a genuinely separate process reads it back

Not a fresh variable in the same kernel — a real `subprocess`, a different interpreter, its own memory space, sharing nothing with this notebook but the filesystem. It is handed the path, reads the JSON, and prints the remembered penalty for one error type.

This is the claim of the whole notebook, and it is the only way to check it that cannot be faked by a leftover variable.

In [ ]:
import subprocess
import sys

CHILD = """
import json, sys
p = sys.argv[1]
items = json.loads(open(p).read())["deductions"]
hit = [i for i in items if i["error_type"] == "missing units"][0]
print("PID", __import__("os").getpid(), "recovered:", hit["error_type"], "->", hit["points_deducted"], "marks")
"""

proc = subprocess.run([sys.executable, "-c", CHILD, str(path)], capture_output=True, text=True)
print("this notebook PID:", __import__("os").getpid())
print(proc.stdout.strip())
if proc.stderr.strip():
    print("stderr:", proc.stderr.strip())

assert proc.returncode == 0, proc.stderr
assert "-> 2.0 marks" in proc.stdout, "the separate process must recover the exact decision"

## Step 8 — the consistency hook

Ported from `memory_hooks.on_deduction_applied`: before saving a new decision, look up what the same error type cost last time, and warn if the new figure disagrees by more than one mark. The warning is *advisory* in the original — it logs and then saves the new value anyway. Ported with that behaviour intact, because it matters in Step 11.

In [ ]:
def apply_deduction(course_id: str, error_type: str, points_deducted: float, reasoning: str,
                    *, instructor_id: str = "") -> dict:
    """Check a new penalty against the remembered one, then record it.

    Returns a small record of what the check found, so the caller can see the
    disagreement rather than only find it in a log.
    """
    past = find_similar_deductions(course_id, error_type, max_results=3)
    result = {"error_type": error_type, "new": points_deducted, "previous": None, "drift": None}
    if past:
        prev = past[0].get("points_deducted")
        try:
            if prev is not None:
                result["previous"] = float(prev)
                result["drift"] = abs(float(prev) - float(points_deducted))
                if result["drift"] > 1:
                    print(f"  CONSISTENCY WARNING: {error_type!r} was {prev}, now {points_deducted}")
        except (TypeError, ValueError):
            pass
    save_deduction(course_id, error_type=error_type, points_deducted=points_deducted,
                   reasoning=reasoning, instructor_id=instructor_id)
    return result

## Step 9 — session two: the same mistake, graded again

A stand-in grader. Given an error type it has seen before, it reuses the remembered penalty; given a new one it falls back to its own default — which, to make the point, is deliberately *different* from what session one decided. Without memory it would apply the default and drift. With memory it applies 2.0 again.

This is what "consistency of judgment across sessions" means concretely: the same error, the same marks, months apart.

In [ ]:
COLD_DEFAULTS = {
    "missing units": 5.0,            # what a cold grader would have said this time
    "sign error in integration": 1.0,
    "arithmetic slip": 1.0,          # genuinely new — no memory of it
}


def grade_with_memory(course_id: str, error_type: str) -> tuple[float, str]:
    """Penalty for an error type: the remembered one if there is one, else the default."""
    past = find_similar_deductions(course_id, error_type, max_results=1)
    if past and past[0].get("error_type", "").lower() == error_type.lower():
        return float(past[0]["points_deducted"]), "from memory"
    return COLD_DEFAULTS.get(error_type, 1.0), "cold default"


rows = []
for et in ("missing units", "sign error in integration", "arithmetic slip"):
    remembered, source = grade_with_memory(COURSE, et)
    rows.append((et, COLD_DEFAULTS.get(et, 1.0), remembered, source))

nbio.table(rows, ("error type", "cold grader would say", "actually applied", "source"))
print()

assert grade_with_memory(COURSE, "missing units") == (2.0, "from memory")
assert grade_with_memory(COURSE, "arithmetic slip") == (1.0, "cold default")
print("The two error types seen in session one cost exactly what they cost then.")
print("The unseen one falls back to a default and becomes the precedent for next time.")

## Step 10 — what the memory looks like after both sessions

The new error type is saved, so session three will find it. `count` on the re-seen types is still 1 — `grade_with_memory` only *reads*; nothing is written unless a decision is applied.

In [ ]:
apply_deduction(COURSE, "arithmetic slip", 1.0, "Correct method, addition slipped in the last line.",
                instructor_id="demo-instructor")

items = json.loads(path.read_text())["deductions"]
nbio.table([(i["error_type"], i["points_deducted"], i["count"]) for i in items],
           ("error type", "marks", "count"))
assert len(items) == 4

## Step 11 — flaw one: the warning does not stop the drift

`apply_deduction` prints a consistency warning and then writes the new value over the old one. So a single disagreeing decision becomes the remembered precedent, and every later submission is graded against it. The memory enforces *agreement with the most recent decision*, not agreement with the first one — which is a weaker property than "the same mistake costs the same marks," and worth knowing before trusting it.

In [ ]:
before = find_similar_deductions(COURSE, "missing units")[0]["points_deducted"]
print(f"remembered before : {before}")

result = apply_deduction(COURSE, "missing units", 5.0, "Regraded, no unit on the final answer.",
                         instructor_id="demo-instructor")

after = find_similar_deductions(COURSE, "missing units")[0]["points_deducted"]
print(f"remembered after  : {after}")
print()
nbio.show_json(result)

assert before == 2.0 and after == 5.0, "the warning is advisory; the new value wins"
assert result["drift"] == 3.0
print()
print("Next session, 'missing units' costs 5 marks, and nothing records that it used to cost 2.")

## Step 12 — flaw two: the cap drops the oldest, not the least used

`items[-MAX_ITEMS:]` keeps the most recently *inserted* entries. An error type recorded early and matched hundreds of times since is evicted ahead of one appended yesterday and never seen again — `count` is stored but never consulted by the eviction. Demonstrated here on a small cap in a throwaway course so the real store is untouched.

In [ ]:
SMALL_COURSE = "DEMO-CAP-TEST"
small_path = store_path(SMALL_COURSE)
if small_path.is_file():
    small_path.unlink()

# Build a store by hand so the cap can be exercised without a 500-entry loop.
entries = [{"error_type": "very common error", "points_deducted": 2.0, "reasoning": "seen constantly",
            "instructor_id": "", "count": 412}]
entries += [{"error_type": f"rare error {i}", "points_deducted": 1.0, "reasoning": "seen once",
             "instructor_id": "", "count": 1} for i in range(4)]

CAP = 4
capped = entries[-CAP:] if len(entries) > CAP else entries

nbio.table([(e["error_type"], e["count"], "KEPT" if e in capped else "EVICTED") for e in entries],
           ("error type", "times seen", "under a cap of 4"))

assert entries[0] not in capped, "the most-used entry is the one evicted"
assert all(e["count"] == 1 for e in capped)
print()
print("The entry with 412 hits is evicted; four entries seen once each survive.")
print("`count` is written on every save and read by nothing.")

## Step 13 — flaw three: the lookup is a substring match, not a meaning match

`find_similar_deductions` is named for similarity and implemented as `q in hay`. A grader that phrases the same mistake differently gets nothing back — and "nothing back" is indistinguishable from "never seen this before," so it silently falls through to a cold default and the inconsistency the memory exists to prevent happens anyway.

This is the gap `03-semantic-recall.ipynb` picks up.

In [ ]:
probes = ["units", "missing units", "no unit given on the answer", "omitted dimensional annotation"]

rows = []
for p_ in probes:
    hits = find_similar_deductions(COURSE, p_)
    rows.append((p_, len(hits), hits[0]["error_type"] if hits else "— nothing —"))

nbio.table(rows, ("query", "hits", "top match"))
print()

assert find_similar_deductions(COURSE, "missing units"), "the exact stored phrase matches"
assert find_similar_deductions(COURSE, "omitted dimensional annotation") == [], \
    "a true paraphrase of the same mistake finds nothing"
print("Exact substring: found. Same mistake, different words: not found.")

## Step 14 — clean up the throwaway store

The course store from Steps 2–13 is left in place on purpose — re-running this notebook rebuilds it from scratch at Step 2, and leaving it makes the "it is a file, go look at it" point checkable. The cap-test store has served its purpose and goes.

In [ ]:
if small_path.is_file():
    small_path.unlink()

print(f"removed: {small_path.relative_to(repo_root)}")
print(f"kept   : {path.relative_to(repo_root)}  ({path.stat().st_size} bytes)")
print()
print("`runs/` is gitignored — nothing written by this notebook can be committed.")

## Where this fits

`01-short-term.ipynb` ends at the boundary of a conversation: pin a fact and it still dies when the session does. This notebook crosses that boundary with the cheapest mechanism that works — a JSON file and a process that reads it.

`03-semantic-recall.ipynb` takes the flaw from Step 13 and addresses it: recall by meaning rather than by exact string, which turns out to be the same machinery as `01-tools/03-embed` pointed at the agent's own past instead of at documents.

## What did not come across

- **Tenant scoping.** The original keys the store path by `get_tenant_id()` as well as `course_id`, so two institutions on the same deployment cannot read each other's memory. Replaced with `course_id` alone — multi-tenancy is a deployment concern, and the isolation mechanism here is the file path.
- **The settings object.** `get_settings().runs_dir_path()` and `settings.trust.deduction_memory_max` become `nbio.runs_dir()` and a module constant. Behaviour identical; only the source of the values differs.
- **The async lifecycle hooks.** `on_grading_start`, `on_rubric_generated`, `on_student_graded` and the rubric store next to the deduction store are the plumbing that calls this memory at the right moments in a grading run. Step 9's `grade_with_memory` is a synchronous stand-in for that call site.
- **The LLM grader itself.** Real grading runs a model against a rubric, extracts an `errors_analysis` block, and feeds each error through `on_deduction_applied`. This notebook supplies the error types directly, so every number printed is reproducible with no key.
- **Concurrency.** `save_deduction` is read-modify-write with no lock. Two graders writing the same course file at once will lose one of the writes. The original carries the same exposure; a production store would need a lock or a real database.